# 03 — Verification Simulations

Three analytical scope-condition verifications: uniform failure, random failure, partial heterogeneity.

In [1]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT / "src"))

OUT_FIGURES = REPO_ROOT / "outputs" / "figures"
OUT_TABLES  = REPO_ROOT / "outputs" / "tables"
OUT_DATA    = REPO_ROOT / "outputs" / "data"
OUT_LOGS    = REPO_ROOT / "outputs" / "logs"
for d in [OUT_FIGURES, OUT_TABLES, OUT_DATA, OUT_LOGS]:
    d.mkdir(parents=True, exist_ok=True)

import json, numpy as np, pandas as pd
from run_verification import run_scenario, SCENARIOS
from run_simulation import SimConfig, load_config, C_GATE, C_COMP_MEAN, C_PERMISSIVE, _style_ax
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

cfg = load_config(str(REPO_ROOT / "src" / "params_default.json"))

In [2]:
# Run all 3 verification scenarios
all_results = {}
for name in SCENARIOS:
    all_results[name] = run_scenario(name, cfg, OUT_DATA)
    print(f"  {SCENARIOS[name][0]}: Gates unsafe = {all_results[name]['Gates']['unsafe_deployment_rate']:.3f}")


Verification Simulation: Uniform Failure



  Method                     Deploy%  Unsafe%  Contam%
  ---------------------------------------------------
  Gates                        0.282    0.000    0.000  [0.000, 0.000]
  Composite (matched)          0.282    0.001    0.004
  Composite (moderate)         0.620    0.012    0.019  [0.006, 0.020]
  Permissive                   0.775    0.040    0.052  [0.029, 0.053]
  Uniform Failure: Gates unsafe = 0.000

Verification Simulation: Random Failure



  Method                     Deploy%  Unsafe%  Contam%
  ---------------------------------------------------
  Gates                        0.307    0.009    0.029  [0.004, 0.015]
  Composite (matched)          0.307    0.009    0.029
  Composite (moderate)         0.675    0.061    0.090  [0.047, 0.077]
  Permissive                   0.833    0.086    0.103  [0.071, 0.104]
  Random Failure: Gates unsafe = 0.009

Verification Simulation: Partial Heterogeneity



  Method                     Deploy%  Unsafe%  Contam%
  ---------------------------------------------------
  Gates                        0.312    0.008    0.026  [0.003, 0.014]
  Composite (matched)          0.312    0.015    0.048
  Composite (moderate)         0.686    0.074    0.108  [0.059, 0.090]
  Permissive                   0.830    0.094    0.113  [0.076, 0.114]
  Partial Heterogeneity: Gates unsafe = 0.008


In [3]:
# Build summary table
rows = []
for name in SCENARIOS:
    r = all_results[name]
    if "error" in r: continue
    label = SCENARIOS[name][0]
    for method in ["Gates", "Composite (moderate)", "Composite (matched)", "Permissive"]:
        m = r[method]
        ci_lo, ci_hi = m.get("unsafe_deployment_rate_ci", (float("nan"), float("nan")))
        rows.append({"Scenario": label, "Decision Rule": method,
            "Deployment Rate": m["deployment_rate"],
            "Unsafe Deployment Rate": m["unsafe_deployment_rate"],
            "Unsafe Deploy 95% CI Low": ci_lo, "Unsafe Deploy 95% CI High": ci_hi,
            "Contamination Rate": m["unsafe_among_deployed"]})
summary_df = pd.DataFrame(rows)
summary_df.to_csv(OUT_DATA / "verification_summary.csv", index=False, lineterminator="\n")
print(summary_df.to_string(index=False))

             Scenario        Decision Rule  Deployment Rate  Unsafe Deployment Rate  Unsafe Deploy 95% CI Low  Unsafe Deploy 95% CI High  Contamination Rate
      Uniform Failure                Gates            0.282                   0.000                  0.000000                   0.000000            0.000000
      Uniform Failure Composite (moderate)            0.620                   0.012                  0.006000                   0.020000            0.019355
      Uniform Failure  Composite (matched)            0.282                   0.001                       NaN                        NaN            0.003546
      Uniform Failure           Permissive            0.775                   0.040                  0.029000                   0.053000            0.051613
       Random Failure                Gates            0.307                   0.009                  0.004000                   0.015025            0.029316
       Random Failure Composite (moderate)            0.67

In [4]:
# Save detailed JSON results
json_results = {}
for name, r in all_results.items():
    if "error" in r:
        json_results[name] = r; continue
    json_results[name] = {}
    for method in ["Gates", "Composite (matched)", "Composite (moderate)", "Permissive"]:
        m = r[method]
        json_results[name][method] = {k: v if not isinstance(v, tuple) else list(v) for k, v in m.items()}
    json_results[name]["gate_deploy_rate"] = r.get("gate_deploy_rate")
(OUT_DATA / "verification_results.json").write_text(json.dumps(json_results, indent=2, default=str), encoding="utf-8", newline="\n")

6029

In [5]:
# Verification comparison figure
scenarios = list(SCENARIOS.keys())
fig, axes = plt.subplots(1, len(scenarios), figsize=(5 * len(scenarios), 4.5), sharey=True)
for ax, name in zip(axes, scenarios):
    r = all_results[name]
    methods = ["Gates", "Composite\n(moderate)", "Composite\n(matched)", "Permissive"]
    keys = ["Gates", "Composite (moderate)", "Composite (matched)", "Permissive"]
    colors = [C_GATE, C_COMP_MEAN, "#67001F", C_PERMISSIVE]
    rates = [r[k]["unsafe_deployment_rate"] for k in keys]
    bars = ax.bar(methods, rates, color=colors, edgecolor="white")
    for bar, val in zip(bars, rates):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height()+0.001, f"{val:.3f}", ha="center", va="bottom", fontsize=8)
    ax.set_title(SCENARIOS[name][0], fontsize=11)
    ax.set_ylabel("Unsafe deployment rate" if name == scenarios[0] else "")
    _style_ax(ax); ax.set_ylim(bottom=0); ax.tick_params(axis="x", labelsize=8)
plt.suptitle("Verification Simulations: Unsafe Deployment Rate by Scenario", fontsize=12, y=1.02)
plt.tight_layout()
fig.savefig(OUT_FIGURES / "fig_verification_comparison.png", dpi=300, bbox_inches="tight")
fig.savefig(OUT_FIGURES / "fig_verification_comparison.pdf", bbox_inches="tight")
plt.close(fig)
print("Verification figure saved.")

C:\Users\Walter.Brown\AppData\Local\Temp\ipykernel_23324\497423496.py:17: UserWarning: Tight layout not applied. The bottom and top margins cannot be made large enough to accommodate all Axes decorations.
  plt.tight_layout()


Verification figure saved.
